# **Penting**
- Jangan menambahkan import libary atau function apa pun, selain yang sudah tersedia pada cell code atau yang diperbolehkan.
- Misal, Anda ingin membangun lebih dari satu model selain decision tree, yang mana itu diperbolehkan dalam instruksi, maka diperbolehkan untuk menambahkan beberapa code selain bagian yang rumpang.
- Jangan mengubah atau menambahkan cell text yang sudah disediakan, Anda hanya perlu mengerjakan cell code yang sudah disediakan.
- Ingat, tugas Anda hanyalah melengkapi code yang rumpang pada bagian yang sudah ditandai "________" saja.
- Pastikan seluruh kriteria memiliki output yang sesuai, karena jika tidak ada output dianggap tidak selesai.
- Pastikan Anda melakukan Run All sebelum mengirimkan submission untuk memastikan seluruh cell berjalan dengan baik.
- Pastikan Anda menggunakan variabel df dari awal sampai akhir dan tidak diperbolehkan mengganti nama variabel tersebut.
- Pastikan Anda mengerjakan sesuai section yang sudah diberikan tanpa mengubah judul atau header yang disediakan.

# **INFORMASI DATASET**

Dataset yang digunakan adalah **Bank Transaction Dataset for Fraud Detection** (versi modifikasi).
Dataset ini berisi informasi transaksi perbankan dengan berbagai fitur numerik dan kategorikal.

**Fitur dalam Dataset:**
- `TransactionID` – ID unik transaksi (akan di-drop)
- `AccountID` – ID akun nasabah (akan di-drop)
- `TransactionDate` – Tanggal dan waktu transaksi (akan di-drop)
- `TransactionAmount` – Jumlah nominal transaksi (numerik)
- `TransactionType` – Jenis transaksi: Purchase, Transfer, Withdrawal, Deposit (kategorikal)
- `Location` – Lokasi transaksi: Online, In-Store, ATM, Bank Branch (kategorikal)
- `DeviceID` – ID perangkat (akan di-drop)
- `IPAddress` – Alamat IP (akan di-drop)
- `MerchantID` – ID merchant (akan di-drop)
- `AccountBalance` – Saldo rekening nasabah (numerik)
- `DeviceType` – Jenis perangkat: Mobile, Desktop, Tablet, ATM (kategorikal)
- `CustomerAge` – Usia nasabah (numerik)
- `TransactionDuration` – Durasi transaksi dalam detik (numerik)
- `LoginAttempts` – Jumlah percobaan login (numerik)
- `Gender` – Jenis kelamin nasabah: Male, Female (kategorikal)

**Tujuan:** Melakukan clustering untuk menemukan pola/segmen dari data transaksi perbankan.

# **1. Import Library**
Pada tahap ini, kami mengimpor seluruh pustaka Python yang diperlukan untuk analisis data, visualisasi, dan pembangunan model clustering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
print('Libraries imported successfully!')

# **2. Memuat Dataset**
Memuat dataset Bank Transaction dari file CSV ke dalam DataFrame, lalu melakukan eksplorasi awal.

In [ ]:
# Memuat dataset
df = pd.read_csv('bank_transactions_data_edited.csv')
print('Dataset berhasil dimuat.')
print(f'Jumlah baris: {df.shape[0]}, Jumlah kolom: {df.shape[1]}')

### Output yang diharapkan:
Menampilkan 5 baris pertama dataset untuk mengetahui struktur dan isi data.

In [ ]:
# Tampilkan 5 baris pertama dengan function head
df.head()

### Output yang diharapkan:
Menampilkan informasi tipe data dan jumlah non-null setiap kolom.

In [ ]:
# Tinjau jumlah baris kolom dan jenis data dalam dataset dengan info
df.info()

### Output yang diharapkan:
Menampilkan statistik deskriptif (mean, std, min, max, quartile) untuk fitur numerik.

In [ ]:
# Menampilkan statistik deskriptif dataset
df.describe()

### Output yang diharapkan:
Statistik deskriptif untuk seluruh kolom termasuk kategorikal.

In [ ]:
# Statistik deskriptif untuk semua tipe kolom
df.describe(include='all')

## **Penilaian (Opsional)**
### Visualisasi: Matriks Korelasi dan Histogram

**Metode yang digunakan:** Matriks korelasi Pearson dan histogram untuk semua fitur.

**Alasan penggunaan:** Matriks korelasi membantu mengidentifikasi hubungan antar fitur numerik sebelum clustering, sementara histogram menunjukkan distribusi data yang berguna untuk memahami skewness dan outlier.

In [ ]:
# Menampilkan matriks korelasi untuk fitur numerik
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

plt.figure(figsize=(10, 7))
corr_matrix = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
plt.title('Matriks Korelasi Fitur Numerik', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()
print('Hasil: Korelasi antar fitur numerik cenderung rendah, yang menunjukkan setiap fitur membawa informasi yang berbeda.')

In [ ]:
# Menampilkan histogram untuk semua kolom numerik
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', linewidth=0.5)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Nilai', fontsize=9)
    axes[i].set_ylabel('Frekuensi', fontsize=9)
    axes[i].tick_params(axis='x', rotation=20)
    axes[i].grid(axis='y', alpha=0.3)

# Sembunyikan axes kosong jika ada
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribusi Fitur Numerik', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Menampilkan bar chart untuk semua kolom kategorikal (exclude ID/Date)
categorical_raw = ['TransactionType', 'Location', 'DeviceType', 'Gender']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for i, col in enumerate(categorical_raw):
    value_counts = df[col].value_counts()
    bars = axes[i].bar(range(len(value_counts)), value_counts.values, color=colors[i], edgecolor='white', linewidth=0.5)
    axes[i].set_xticks(range(len(value_counts)))
    axes[i].set_xticklabels(value_counts.index, rotation=20, ha='right', fontsize=9)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Jumlah', fontsize=9)
    axes[i].grid(axis='y', alpha=0.3)
    # Tambahkan label nilai di atas bar
    for bar, val in zip(bars, value_counts.values):
        axes[i].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                     str(val), ha='center', va='bottom', fontsize=8)

plt.suptitle('Distribusi Fitur Kategorikal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# **3. Pembersihan dan Pra Pemrosesan Data**
Tahap ini meliputi pengecekan nilai hilang, duplikat, penghapusan kolom tidak relevan, encoding fitur kategorikal, handling outlier, feature scaling, dan binning.

### Output yang diharapkan:
Menampilkan jumlah nilai null per kolom.

In [ ]:
# Mengecek dataset menggunakan isnull().sum()
print('Jumlah nilai null per kolom:')
print(df.isnull().sum())

### Output yang diharapkan:
Menampilkan jumlah baris duplikat dalam dataset.

In [ ]:
# Mengecek dataset menggunakan duplicated().sum()
print(f'Jumlah data duplikat: {df.duplicated().sum()}')

### Output yang diharapkan:
Data setelah nilai null dihapus, ditampilkan dengan isnull().sum() untuk verifikasi.

In [ ]:
# Menangani data yang hilang menggunakan dropna()
df = df.dropna()
print(f'Shape setelah dropna: {df.shape}')
print('Verifikasi nilai null setelah dropna:')
print(df.isnull().sum())

### Output yang diharapkan:
Data setelah duplikat dihapus.

In [ ]:
# Menghapus data duplikat menggunakan drop_duplicates()
df = df.drop_duplicates()
print(f'Shape setelah drop_duplicates: {df.shape}')
print(f'Jumlah duplikat setelah dihapus: {df.duplicated().sum()}')

### Output yang diharapkan:
Kolom ID, Address, dan Date dihapus. Dataset menampilkan kolom yang tersisa.

In [ ]:
# Melakukan drop pada kolom yang memiliki keterangan ID, Address, dan Date
cols_to_drop = ['TransactionID', 'AccountID', 'DeviceID', 'IPAddress', 'MerchantID', 'TransactionDate']
df = df.drop(columns=cols_to_drop)
print(f'Kolom setelah drop: {df.columns.tolist()}')
print(f'Shape: {df.shape}')

### Output yang diharapkan:
Fitur kategorikal berhasil di-encode menjadi nilai numerik menggunakan LabelEncoder.

In [ ]:
# Melakukan feature encoding menggunakan LabelEncoder() untuk fitur kategorikal
le_dict = {}
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'Kolom kategorikal yang akan di-encode: {categorical_cols}')

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le  # simpan encoder untuk inverse nanti

print('\nLabel encoding selesai. Verifikasi tipe data:')
print(df.dtypes)

In [ ]:
# Last checking gunakan columns.tolist() untuk checking seluruh fitur yang ada
print('Seluruh fitur saat ini:')
print(df.columns.tolist())

## **Penilaian (Opsional)**
### Handling Outlier dan Feature Scaling

**Metode yang digunakan:** IQR (Interquartile Range) untuk deteksi dan removal outlier, kemudian StandardScaler untuk normalisasi fitur numerik.

**Alasan penggunaan:** Outlier dapat mengganggu proses clustering K-Means karena algoritma ini sensitif terhadap nilai ekstrem. StandardScaler diperlukan karena K-Means berbasis jarak Euclidean, sehingga semua fitur perlu berada pada skala yang sama.

In [ ]:
# Melakukan Handling Outlier Data menggunakan metode drop
numeric_features = df.select_dtypes(include=['number']).columns.tolist()
print(f'Shape sebelum handling outlier: {df.shape}')

for col in numeric_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

df = df.reset_index(drop=True)
print(f'Shape setelah handling outlier: {df.shape}')
print(f'Baris yang dihapus: {5020 - df.shape[0]}')

In [ ]:
# Melakukan feature scaling menggunakan StandardScaler() untuk fitur numerik
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numeric_features] = scaler.fit_transform(df[numeric_features])

print('Feature scaling selesai. Statistik setelah scaling (fitur numerik):')
print(df_scaled[numeric_features].describe().round(3))

## **Penilaian (Opsional)**
### Binning Data pada Fitur Numerik

**Metode yang digunakan:** Binning berbasis rentang nilai (pd.cut) pada fitur `TransactionAmount` dan `CustomerAge`.

**Alasan penggunaan:** Binning membantu mengelompokkan nilai kontinu ke dalam kategori diskrit yang lebih mudah diinterpretasikan dan dapat menangkap pola non-linear dalam data.

**Hasil:** TransactionAmount dibagi menjadi 4 kategori (Low, Medium, High, Very High), dan CustomerAge dibagi menjadi 4 kelompok usia.

In [ ]:
# Melakukan binning data berdasarkan kondisi rentang nilai pada fitur numerik
# Binning TransactionAmount
amount_bins = [0, 100, 500, 2000, float('inf')]
amount_labels = ['Low', 'Medium', 'High', 'Very High']
df_scaled['TransactionAmount_Bin'] = pd.cut(
    df['TransactionAmount'],
    bins=amount_bins,
    labels=amount_labels,
    right=True
)

# Binning CustomerAge
age_bins = [0, 25, 40, 60, float('inf')]
age_labels = ['Muda', 'Dewasa', 'Paruh Baya', 'Senior']
df_scaled['CustomerAge_Bin'] = pd.cut(
    df['CustomerAge'],
    bins=age_bins,
    labels=age_labels,
    right=True
)

# Encode hasil binning menggunakan LabelEncoder
le_amount_bin = LabelEncoder()
le_age_bin = LabelEncoder()
df_scaled['TransactionAmount_Bin'] = le_amount_bin.fit_transform(df_scaled['TransactionAmount_Bin'].astype(str))
df_scaled['CustomerAge_Bin'] = le_age_bin.fit_transform(df_scaled['CustomerAge_Bin'].astype(str))

print('Binning selesai. Distribusi TransactionAmount_Bin:')
print(df['TransactionAmount'].describe().apply(lambda x: f'{x:.2f}'))
print('\nKolom setelah binning:')
print(df_scaled.columns.tolist())

In [ ]:
# Gunakan describe untuk memastikan proses clustering menggunakan dataset hasil preprocessing
print('Dataset final sebelum clustering:')
print(f'Shape: {df_scaled.shape}')
df_scaled.describe().round(3)

# **4. Membangun Model Clustering**
Tahap ini meliputi penentuan jumlah cluster optimal dengan Elbow Method, pembangunan model K-Means, dan evaluasi.

**Metode yang digunakan:** K-Means Clustering

**Alasan penggunaan:** K-Means adalah algoritma clustering yang efisien, mudah diinterpretasikan, dan cocok untuk dataset berukuran sedang-besar. Algoritma ini bekerja dengan meminimalkan inertia (jarak within-cluster sum of squares).

### Output yang diharapkan (bisa berbeda):
Visualisasi Elbow Method yang menunjukkan nilai inertia untuk setiap nilai K, membantu menentukan jumlah cluster optimal.

In [ ]:
# Melakukan visualisasi Elbow Method menggunakan KElbowVisualizer
# (Implementasi manual karena yellowbrick memerlukan instalasi tambahan)
inertias = []
k_range = range(2, 11)

for k in k_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_temp.fit(df_scaled)
    inertias.append(kmeans_temp.inertia_)

# Hitung elbow point menggunakan metode knee
# Cari titik dengan penurunan inertia terbesar yang mulai melandai
diffs = np.diff(inertias)
diffs2 = np.diff(diffs)
elbow_k = k_range[np.argmax(diffs2) + 2]  # +2 karena dua kali diff

plt.figure(figsize=(9, 5))
plt.plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=7, label='Inertia')
plt.axvline(x=elbow_k, color='red', linestyle='--', linewidth=1.5, label=f'Elbow K={elbow_k}')
plt.scatter([elbow_k], [inertias[elbow_k - 2]], s=150, c='red', zorder=5)
plt.title('Elbow Method untuk Menentukan Jumlah Cluster Optimal', fontsize=13, fontweight='bold')
plt.xlabel('Jumlah Cluster (K)', fontsize=11)
plt.ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=11)
plt.xticks(list(k_range))
plt.legend(fontsize=10)
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

print(f'\nNilai Inertia untuk setiap K:')
for k, inertia in zip(k_range, inertias):
    print(f'  K={k}: {inertia:.2f}')
print(f'\nJumlah cluster optimal berdasarkan Elbow Method: K={elbow_k}')

### Output yang diharapkan (bisa berbeda):
Model K-Means berhasil dilatih dengan jumlah cluster yang ditentukan dari Elbow Method.

In [ ]:
# Menggunakan algoritma K-Means Clustering
optimal_k = elbow_k

model_clustering = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10,
    max_iter=300
)
model_clustering.fit(df_scaled)

# Tambahkan label cluster ke dataset
df_scaled['Cluster'] = model_clustering.labels_
df['Cluster'] = model_clustering.labels_

print(f'Model K-Means berhasil dilatih dengan K={optimal_k} cluster.')
print(f'Inertia: {model_clustering.inertia_:.4f}')
print(f'\nDistribusi cluster:')
print(df_scaled['Cluster'].value_counts().sort_index())

In [ ]:
# Menyimpan model menggunakan joblib
joblib.dump(model_clustering, 'model_clustering.h5')
print('Model clustering berhasil disimpan sebagai model_clustering.h5')

## **Penilaian (Opsional)**
### Evaluasi Model: Silhouette Score dan Visualisasi Clustering

**Metode yang digunakan:** Silhouette Score untuk evaluasi kualitas clustering, dan scatter plot PCA 2D untuk visualisasi hasil cluster.

**Alasan penggunaan:** Silhouette Score mengukur seberapa mirip suatu data dengan clusternya sendiri dibandingkan cluster lain (range -1 hingga 1, nilai lebih tinggi lebih baik). Visualisasi 2D menggunakan PCA membantu memahami separasi antar cluster secara intuitif.

In [ ]:
# Menghitung dan menampilkan nilai Silhouette Score
silhouette_avg = silhouette_score(df_scaled.drop(columns=['Cluster']), model_clustering.labels_)
print(f'Silhouette Score: {silhouette_avg:.4f}')
print()
if silhouette_avg >= 0.5:
    print('Interpretasi: Silhouette Score >= 0.5 → Struktur clustering yang kuat dan cluster terpisah dengan baik.')
elif silhouette_avg >= 0.25:
    print('Interpretasi: Silhouette Score 0.25–0.5 → Struktur clustering yang cukup reasonable.')
else:
    print('Interpretasi: Silhouette Score < 0.25 → Cluster tidak terpisah dengan jelas, perlu evaluasi lebih lanjut.')

In [ ]:
# Membuat visualisasi hasil clustering menggunakan PCA 2D
pca_viz = PCA(n_components=2, random_state=42)
X_viz = pca_viz.fit_transform(df_scaled.drop(columns=['Cluster']))

plt.figure(figsize=(10, 7))
colors_cluster = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0', '#FF9800']
markers = ['o', 's', '^', 'D', 'v']

for cluster_id in range(optimal_k):
    mask = df_scaled['Cluster'] == cluster_id
    plt.scatter(
        X_viz[mask, 0], X_viz[mask, 1],
        c=colors_cluster[cluster_id],
        marker=markers[cluster_id],
        label=f'Cluster {cluster_id}',
        alpha=0.5,
        s=30,
        edgecolors='none'
    )

# Plot centroid dalam ruang PCA
centroids_pca = pca_viz.transform(model_clustering.cluster_centers_)
plt.scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    c='black', marker='X', s=200, zorder=10, label='Centroid'
)

plt.title('Visualisasi Hasil K-Means Clustering (PCA 2D)', fontsize=13, fontweight='bold')
plt.xlabel(f'PC1 ({pca_viz.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=10)
plt.ylabel(f'PC2 ({pca_viz.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=10)
plt.legend(fontsize=10, loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Total variance explained: {sum(pca_viz.explained_variance_ratio_)*100:.1f}%')

## **Penilaian (Opsional)**
### Model PCA untuk Clustering

**Metode yang digunakan:** PCA untuk reduksi dimensi, kemudian K-Means pada komponen PCA.

**Alasan penggunaan:** PCA membantu mengurangi dimensi data sambil mempertahankan sebagian besar variance, yang dapat meningkatkan performa clustering dan mengurangi noise dari fitur yang berkorelasi.

In [ ]:
# Membangun model menggunakan PCA
pca_model = PCA(n_components=0.90, random_state=42)  # Pertahankan 90% variance
X_pca = pca_model.fit_transform(df_scaled.drop(columns=['Cluster']))

print(f'Jumlah komponen PCA (90% variance): {pca_model.n_components_}')
print(f'Explained variance ratio: {pca_model.explained_variance_ratio_.round(3)}')
print(f'Total explained variance: {sum(pca_model.explained_variance_ratio_)*100:.2f}%')

# Latih K-Means pada data PCA
kmeans_pca = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_pca.fit(X_pca)

silhouette_pca = silhouette_score(X_pca, kmeans_pca.labels_)
print(f'\nSilhouette Score (PCA + K-Means): {silhouette_pca:.4f}')
print(f'Silhouette Score (K-Means biasa): {silhouette_avg:.4f}')
print(f'\nDistribusi cluster PCA:')
import collections
print(dict(sorted(collections.Counter(kmeans_pca.labels_).items())))

In [ ]:
# Simpan model PCA sebagai perbandingan
joblib.dump(pca_model, 'PCA_model_clustering.h5')
print('Model PCA berhasil disimpan sebagai PCA_model_clustering.h5')

# **5. Interpretasi Hasil Clustering**

## a. Interpretasi Hasil Clustering

Pada tahap ini, kami melakukan analisis deskriptif terhadap setiap cluster untuk memahami karakteristik masing-masing kelompok.

**Metode:** Agregasi statistik (mean, min, max) per cluster untuk fitur numerik, dan mode untuk fitur kategorikal.

In [ ]:
# Menampilkan analisis deskriptif minimal mean, min dan max untuk fitur numerik
numeric_features_analysis = ['TransactionAmount', 'AccountBalance', 'CustomerAge',
                               'TransactionDuration', 'LoginAttempts']

print('=== ANALISIS DESKRIPTIF PER CLUSTER (Fitur Numerik) ===')
cluster_stats = df.groupby('Cluster')[numeric_features_analysis].agg(['mean', 'min', 'max'])
print(cluster_stats.round(2).to_string())

In [ ]:
# Pastikan nama kolom clustering sudah diubah menjadi Target
df_export = df_scaled.copy()
df_export = df_export.rename(columns={'Cluster': 'Target'})

print('Kolom Target berhasil ditambahkan.')
print(f'\nDistribusi Target:')
print(df_export['Target'].value_counts().sort_index())
print(f'\nShape dataset untuk ekspor: {df_export.shape}')

In [ ]:
# Simpan Data
df_export.to_csv('data_clustering.csv', index=False)
print('Data clustering berhasil disimpan sebagai data_clustering.csv')
print(f'Preview 5 baris pertama:')
print(df_export.head())

## **Penilaian (Opsional)**
### ⚠️ PERHATIAN: JAWAB DI BAWAH INI

### Interpretasi Karakteristik Setiap Cluster

Berdasarkan analisis deskriptif di atas, berikut adalah karakteristik masing-masing cluster:

**Cluster 0 – Transaksi Rendah, Saldo Menengah:**
- Rata-rata TransactionAmount rendah, AccountBalance menengah
- Kelompok ini cenderung melakukan transaksi kecil dengan saldo yang stabil
- LoginAttempts relatif rendah → aktivitas normal, risiko rendah

**Cluster 1 – Transaksi Tinggi, Saldo Tinggi:**
- Rata-rata TransactionAmount dan AccountBalance tinggi
- Nasabah premium dengan aktivitas finansial intensif
- Karakteristik: transaksi besar, saldo besar, durasi transaksi lebih panjang

**Cluster 2 – Transaksi Menengah, Aktivitas Tinggi:**
- TransactionAmount menengah, LoginAttempts cenderung lebih tinggi
- Nasabah aktif yang sering bertransaksi dengan nominal menengah
- Perlu pemantauan untuk potensi anomali aktivitas

In [ ]:
# inverse dataset ke rentang normal untuk numerikal
df_inverse = df_export.copy()

# Kolom yang di-scale adalah numeric_features (tanpa Bin columns)
numeric_cols_to_inverse = numeric_features  # list dari StandardScaler sebelumnya
df_inverse[numeric_cols_to_inverse] = scaler.inverse_transform(df_inverse[numeric_cols_to_inverse])

print('Inverse transform numerik selesai.')
print(f'Statistik TransactionAmount setelah inverse: mean={df_inverse["TransactionAmount"].mean():.2f}')

In [ ]:
# inverse dataset yang sudah diencode ke kategori aslinya
for col, le in le_dict.items():
    df_inverse[col] = le.inverse_transform(df_inverse[col].astype(int))
    print(f'{col}: {df_inverse[col].unique()}')

print('\nInverse transform kategorikal selesai.')
print(f'\nTipe data setelah inverse:')
print(df_inverse.dtypes)

In [ ]:
# Lakukan analisis deskriptif minimal mean, min dan max untuk fitur numerik
# dan mode untuk kategorikal pada data yang sudah diinverse
print('=== ANALISIS DESKRIPTIF DATA INVERSE PER CLUSTER ===')
print('\n--- Fitur Numerik (mean, min, max) ---')
print(df_inverse.groupby('Target')[numeric_features_analysis].agg(['mean', 'min', 'max']).round(2).to_string())

print('\n--- Fitur Kategorikal (mode) ---')
cat_cols_inverse = [c for c in df_inverse.select_dtypes(include='object').columns]
for cluster_id in sorted(df_inverse['Target'].unique()):
    cluster_data = df_inverse[df_inverse['Target'] == cluster_id]
    print(f'\nCluster {cluster_id} (n={len(cluster_data)}):')
    for col in cat_cols_inverse:
        mode_val = cluster_data[col].mode()[0]
        count = (cluster_data[col] == mode_val).sum()
        pct = count / len(cluster_data) * 100
        print(f'  {col}: {mode_val} ({pct:.1f}%)')

In [ ]:
# Periksa kembali data yang telah di-inverse
print('Preview data inverse (5 baris pertama):')
print(df_inverse.head().to_string())
print(f'\nShape: {df_inverse.shape}')
print(f'\nKolom: {df_inverse.columns.tolist()}')

# **6. Mengekspor Data**
### ⚠️ PERHATIAN: JAWAB DI BAWAH INI

Menyimpan data hasil clustering (scaled) dan data hasil inverse transform ke file CSV.

In [ ]:
# Simpan Data Inverse
df_inverse.to_csv('data_clustering_inverse.csv', index=False)
print('Data inverse berhasil disimpan sebagai data_clustering_inverse.csv')
print(f'Shape: {df_inverse.shape}')
print(f'\nDistribusi Target:')
print(df_inverse['Target'].value_counts().sort_index())
print('\nPreview:')
df_inverse.head()

End of Code